In [1]:
%pip install spacy
!python -m spacy download en_core_web_md

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_md')



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import spacy
from spacy.matcher import Matcher
from collections import defaultdict

nlp = spacy.load("en_core_web_md")

print("spaCy model loaded successfully!")
print("Customer Support Ticket Routing System")

spaCy model loaded successfully!
Customer Support Ticket Routing System


In [3]:
CATEGORY_EXAMPLES = {

    "Billing Team": [
        "My payment failed",
        "My payment was declined",
        "I was charged twice",
        "There is a transaction problem",
        "My card payment is not working"
    ],

    "Technical Team": [
        "Unable to login to my account",
        "My password is not working",
        "The application is crashing",
        "The website is not working",
        "I am getting a technical error"
    ],

    "Refund Team": [
        "I need a refund",
        "Please refund my order",
        "I want my money back",
        "I want to return my product",
        "I want to cancel and get a refund"
    ],

    "Sales Team": [
        "I want to buy a product",
        "What is the price of this product",
        "Are there any discounts available",
        "Tell me about your offers",
        "What products do you sell"
    ]
}

print("Department categories created successfully!\n")

for department, examples in CATEGORY_EXAMPLES.items():
    print(f"{department}: {len(examples)} reference tickets")

Department categories created successfully!

Billing Team: 5 reference tickets
Technical Team: 5 reference tickets
Refund Team: 5 reference tickets
Sales Team: 5 reference tickets


In [4]:
# Convert reference sentences into spaCy documents

CATEGORY_DOCS = {}

for department, examples in CATEGORY_EXAMPLES.items():
    CATEGORY_DOCS[department] = [
        nlp(example) for example in examples
    ]

print("Reference documents created successfully!\n")

for department, docs in CATEGORY_DOCS.items():
    print(f"{department}: {len(docs)} documents")

Reference documents created successfully!

Billing Team: 5 documents
Technical Team: 5 documents
Refund Team: 5 documents
Sales Team: 5 documents


In [5]:
# Create spaCy Matcher
matcher = Matcher(nlp.vocab)

# Define patterns for each department
PATTERNS = {
    "BILLING": [
        [{"LOWER": {"IN": ["payment", "transaction", "billing"]}}],
        [{"LOWER": "charged"}, {"LOWER": "twice"}],
        [{"LOWER": "payment"},
         {"LOWER": {"IN": ["failed", "declined", "rejected"]}}]
    ],

    "TECHNICAL": [
        [{"LOWER": {"IN": ["login", "password", "signin"]}}],
        [{"LOWER": {"IN": ["crashing", "error", "bug"]}}],
        [{"LOWER": "not"},
         {"LOWER": {"IN": ["working", "opening"]}}]
    ],

    "REFUND": [
        [{"LOWER": "refund"}],
        [{"LOWER": "money"}, {"LOWER": "back"}],
        [{"LOWER": "return"}]
    ],

    "SALES": [
        [{"LOWER": {"IN": ["buy", "purchase", "product"]}}],
        [{"LOWER": {"IN": ["price", "discount", "offer", "offers"]}}]
    ]
}

# Register patterns with the Matcher
for category, patterns in PATTERNS.items():
    matcher.add(category, patterns)

print("Matcher patterns registered successfully!")

for category, patterns in PATTERNS.items():
    print(f"{category}: {len(patterns)} patterns")

Matcher patterns registered successfully!
BILLING: 3 patterns
TECHNICAL: 3 patterns
REFUND: 3 patterns
SALES: 2 patterns


In [6]:
# Sample customer ticket
ticket = "I was charged twice for my order."

# Process the ticket using spaCy
doc = nlp(ticket)

# Find matching patterns
matches = matcher(doc)

print("Customer Ticket:", ticket)
print("\nMatched Patterns:")

for match_id, start, end in matches:
    category = nlp.vocab.strings[match_id]
    matched_text = doc[start:end].text

    print(f"{category}: {matched_text}")

Customer Ticket: I was charged twice for my order.

Matched Patterns:
BILLING: charged twice


In [7]:
# Function to calculate similarity with a department
def calculate_similarity(ticket_doc, department):

    scores = []

    for reference_doc in CATEGORY_DOCS[department]:
        score = ticket_doc.similarity(reference_doc)
        scores.append(score)

    return max(scores) if scores else 0.0


# Test ticket
ticket = "My transaction was declined"
ticket_doc = nlp(ticket)

print("Customer Ticket:", ticket)
print("\nSimilarity Scores:\n")

for department in CATEGORY_DOCS:
    score = calculate_similarity(ticket_doc, department)
    print(f"{department}: {score:.4f}")

Customer Ticket: My transaction was declined

Similarity Scores:

Billing Team: 1.0000
Technical Team: 0.7644
Refund Team: 0.7485
Sales Team: 0.7911


In [8]:
def calculate_all_scores(ticket):

    doc = nlp(ticket)

    # Get semantic similarity scores
    similarity_scores = {
        department: calculate_similarity(doc, department)
        for department in CATEGORY_DOCS
    }

    # Get Matcher results
    matches = matcher(doc)

    matcher_results = {}

    for match_id, start, end in matches:
        category = nlp.vocab.strings[match_id]
        matched_text = doc[start:end].text

        matcher_results.setdefault(category, []).append(matched_text)

    # Combine scores
    final_scores = {}

    for department in CATEGORY_DOCS:

        matcher_name = department.replace(" Team", "").upper()

        match_count = len(set(matcher_results.get(matcher_name, [])))
        bonus = min(match_count * 0.05, 0.15)

        final_scores[department] = min(
            similarity_scores[department] + bonus,
            1.0
        )

    return similarity_scores, matcher_results, final_scores


# Test the combined routing logic
ticket = "My transaction was declined"

similarity, matches, final_scores = calculate_all_scores(ticket)

print("Customer Ticket:", ticket)

print("\nMatcher Results:")
print(matches)

print("\nFinal Routing Scores:")

for department, score in sorted(
    final_scores.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{department}: {score:.4f}")

Customer Ticket: My transaction was declined

Matcher Results:
{'BILLING': ['transaction']}

Final Routing Scores:
Billing Team: 1.0000
Sales Team: 0.7911
Technical Team: 0.7644
Refund Team: 0.7485


In [9]:
# Sample customer ticket
ticket = "I need a refund of 500 rupees from Amazon tomorrow."

# Process the ticket
doc = nlp(ticket)

print("Customer Ticket:", ticket)
print("\nNamed Entities:\n")

if doc.ents:
    for ent in doc.ents:
        print(
            f"Text: {ent.text} | "
            f"Label: {ent.label_} | "
            f"Meaning: {spacy.explain(ent.label_)}"
        )
else:
    print("No named entities detected.")

Customer Ticket: I need a refund of 500 rupees from Amazon tomorrow.

Named Entities:

Text: 500 rupees | Label: QUANTITY | Meaning: Measurements, as of weight or distance
Text: Amazon | Label: ORG | Meaning: Companies, agencies, institutions, etc.
Text: tomorrow | Label: DATE | Meaning: Absolute or relative dates or periods


In [10]:
def route_ticket(ticket):

    # 1. NLP Pipeline
    doc = nlp(ticket)

    # 2. Text Similarity
    similarity_scores = {
        department: calculate_similarity(doc, department)
        for department in CATEGORY_DOCS
    }

    # 3. Matcher
    matches = matcher(doc)
    matcher_results = {}

    for match_id, start, end in matches:
        category = nlp.vocab.strings[match_id]
        matched_text = doc[start:end].text

        matcher_results.setdefault(category, []).append(matched_text)

    # 4. Combine scores
    final_scores = {}

    for department in CATEGORY_DOCS:
        matcher_name = department.replace(" Team", "").upper()

        unique_matches = set(matcher_results.get(matcher_name, []))
        bonus = min(len(unique_matches) * 0.05, 0.15)

        final_scores[department] = min(
            similarity_scores[department] + bonus,
            1.0
        )

    # Rank departments
    ranked = sorted(
        final_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    best_department, best_score = ranked[0]
    second_score = ranked[1][1]

    # Basic uncertainty handling
    if best_score < 0.50 or best_score - second_score < 0.03:
        department = "Manual Review"
    else:
        department = best_department

    # 5. Named Entity Recognition
    entities = [
        {
            "text": ent.text,
            "label": ent.label_,
            "meaning": spacy.explain(ent.label_)
        }
        for ent in doc.ents
    ]

    return {
        "ticket": ticket,
        "department": department,
        "routing_score": round(best_score * 100, 2),
        "similarity_scores": similarity_scores,
        "matcher_results": matcher_results,
        "entities": entities
    }


print("Ticket routing function created successfully!")

Ticket routing function created successfully!


In [11]:
test_tickets = [
    "My payment failed",
    "Unable to login to my account",
    "I need a refund for my order",
    "What discounts are available?",
    "I need a refund of 500 rupees from Amazon tomorrow.",
    "What is the weather today?"
]

for ticket in test_tickets:
    result = route_ticket(ticket)

    print("\n" + "=" * 50)
    print("Ticket:", result["ticket"])
    print("Department:", result["department"])
    print("Routing score:", result["routing_score"], "%")
    print("Matcher:", result["matcher_results"])
    print("Entities:", result["entities"])


Ticket: My payment failed
Department: Billing Team
Routing score: 100.0 %
Matcher: {'BILLING': ['payment', 'payment failed']}
Entities: []

Ticket: Unable to login to my account
Department: Technical Team
Routing score: 100.0 %
Matcher: {'TECHNICAL': ['login']}
Entities: []

Ticket: I need a refund for my order
Department: Refund Team
Routing score: 99.38 %
Matcher: {'REFUND': ['refund']}
Entities: []

Ticket: What discounts are available?
Department: Sales Team
Routing score: 95.69 %
Matcher: {}
Entities: []

Ticket: I need a refund of 500 rupees from Amazon tomorrow.
Department: Refund Team
Routing score: 91.12 %
Matcher: {'REFUND': ['refund']}
Entities: [{'text': '500 rupees', 'label': 'QUANTITY', 'meaning': 'Measurements, as of weight or distance'}, {'text': 'Amazon', 'label': 'ORG', 'meaning': 'Companies, agencies, institutions, etc.'}, {'text': 'tomorrow', 'label': 'DATE', 'meaning': 'Absolute or relative dates or periods'}]

Ticket: What is the weather today?
Department: Manual